# 09_rag_failure_modes_and_debugging: Lost-in-the-Middle

This notebook demonstrates the Lost-in-the-Middle retrieval position bias using text paragraphs scraped from Wikipedia.


In [1]:
import requests
from bs4 import BeautifulSoup

# 1. Scrape corpus
url = "https://en.wikipedia.org/wiki/Retrieval-augmented_generation"
headers = {"User-Agent": "Mozilla/5.0"}
resp = requests.get(url, headers=headers)
soup = BeautifulSoup(resp.content, "html.parser")
corpus = [p.get_text().strip() for p in soup.find_all("p") if len(p.get_text().strip()) > 100][:5]

# Highlight our target document
target = "Lewis et al. (2020) Facebook AI Research introduced RAG."
context_chunks = [corpus[0], corpus[1], corpus[2], target, corpus[3]]

# 2. Re-ordering algorithm
def reorder_context(chunks):
    # Sorts to place high-relevance chunks at outer boundaries
    sorted_chunks = sorted(chunks, key=lambda x: len(x)) # Sort mock relevance by length
    reordered = [None] * len(sorted_chunks)
    left, right = 0, len(sorted_chunks) - 1
    for idx, item in enumerate(sorted_chunks):
        if idx % 2 == 0:
            reordered[left] = item
            left += 1
        else:
            reordered[right] = item
            right -= 1
    return reordered

reordered = reorder_context(context_chunks)
print("=== Original Chunk Order ===")
for i, c in enumerate(context_chunks):
    print(f"Chunk {i}: {c[:60]}...")

print("\n=== Reordered Chunk Order ===")
for i, c in enumerate(reordered):
    print(f"Chunk {i}: {c[:60]}...")


=== Original Chunk Order ===
Chunk 0: Retrieval-augmented generation (RAG) is a technique that ena...
Chunk 1: RAG improves LLMs by incorporating information retrieval bef...
Chunk 2: RAG also reduces the need to retrain LLMs with new data, sav...
Chunk 3: Lewis et al. (2020) Facebook AI Research introduced RAG....
Chunk 4: The term retrieval-augmented generation (RAG) was introduced...

=== Reordered Chunk Order ===
Chunk 0: Lewis et al. (2020) Facebook AI Research introduced RAG....
Chunk 1: RAG also reduces the need to retrain LLMs with new data, sav...
Chunk 2: Retrieval-augmented generation (RAG) is a technique that ena...
Chunk 3: RAG improves LLMs by incorporating information retrieval bef...
Chunk 4: The term retrieval-augmented generation (RAG) was introduced...


### Output Explanation
- The original list places the target RAG origin chunk in the middle.
- The reordering preprocessor shifts the chunks, aligning the most relevant text segments at the start and end of the context prompt payload.
